# Comparing Orchestration Philosophies

Frameworks are products; **orchestration philosophies** are the deeper design choices underneath them. Two teams can both use graphs, or both use chat — and still differ on **who owns routing**, **where state lives**, and **when the topology is fixed**.

This notebook compares five philosophies on the **same ReleaseFlow task** (produce a v2.4.0 customer announcement):

1. **Explicit graph** — compile-time DAG; engineer owns every edge.
2. **Emergent conversation** — runtime dialogue; a manager selects speakers.
3. **Task decomposition** — work as a queue of role-owned tasks.
4. **Supervisor routing** — a meta-agent delegates to workers each turn.
5. **Plan-and-execute** — planner emits a step list; executor follows it.

Each philosophy gets a plain-Python implementation, measurable traits, and trade-off analysis. No frameworks or prior notebooks are required.

In [ ]:
"""
ReleaseFlow — orchestration philosophy comparison environment.
"""

import json
import os
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


CHANGELOG: list[dict[str, str]] = [
    {"version": "2.4.0", "item": "Added bulk export endpoint for reports"},
    {"version": "2.4.0", "item": "Fixed session timeout on mobile browsers"},
    {"version": "2.4.0", "item": "Deprecated legacy v1 webhook format"},
]

RUNBOOKS: list[dict[str, str]] = [
    {"id": "comm-201", "title": "Customer communication standards",
     "text": "Lead with user benefit, list breaking changes separately, include support channel #releases."},
]

STYLE_GUIDE = {"must_include": ["version number", "breaking changes section", "support channel"], "max_words": 200}

print(f"ReleaseFlow: {len(CHANGELOG)} changelog items")

---

## 1. Philosophy vs Framework

| Layer | Example | Question it answers |
|-------|---------|---------------------|
| **Philosophy** | Explicit graph | *Who decides what runs next?* |
| **Pattern** | Sequential pipeline, supervisor | *What shape does coordination take?* |
| **Framework** | LangGraph, AutoGen, CrewAI | *What API implements the pattern?* |

One philosophy can map to multiple frameworks. **Explicit graph** appears in LangGraph, AWS Step Functions + LLM nodes, and hand-rolled Python state machines. Choosing a philosophy comes **before** choosing a vendor.

---

## 2. The Five Philosophies — Overview

| Philosophy | Control locus | Topology fixed when? | Routing mechanism |
|------------|---------------|----------------------|-------------------|
| **Explicit graph** | Engineer | Design / deploy time | Graph edges + condition functions |
| **Emergent conversation** | Manager agent or selector | Runtime (each turn) | Speaker selection from chat history |
| **Task decomposition** | Crew process definition | Task list authoring time | Sequential or hierarchical task queue |
| **Supervisor routing** | Supervisor LLM (simulated) | Each delegation turn | Supervisor picks next worker |
| **Plan-and-execute** | Planner LLM (simulated) | Plan phase, then fixed | Executor walks planned steps |

```
Explicit graph:     [A]──►[B]──►[C]           (edges are code)
Conversation:       A ◄──► B ◄──► C           (manager picks speaker)
Task queue:         T1 → T2 → T3              (tasks carry context)
Supervisor:              [S]──► worker         (S re-evaluates each turn)
Plan-and-execute:   PLAN ──► [s1,s2,s3]       (plan once, execute list)
```

---

## 3. Comparison Axes

Score each philosophy on these axes (implemented in code later):

| Axis | High score means |
|------|------------------|
| **Determinism** | Same inputs → same step order |
| **Auditability** | Easy to explain who ran when and why |
| **Flexibility** | Can adapt path to unexpected situations |
| **Predictable cost** | Bounded LLM calls / turns |
| **Debuggability** | Failures localize to a named step |
| **Time-to-prototype** | Fast to get a first working pipeline |

In [ ]:
class Philosophy(str, Enum):
    EXPLICIT_GRAPH = "explicit_graph"
    CONVERSATION = "conversation"
    TASK_DECOMPOSITION = "task_decomposition"
    SUPERVISOR = "supervisor"
    PLAN_EXECUTE = "plan_execute"


@dataclass
class PhilosophyTraits:
    name: Philosophy
    determinism: int       # 1-5
    auditability: int
    flexibility: int
    predictable_cost: int
    debuggability: int
    time_to_prototype: int
    control_locus: str
    typical_failure_mode: str


TRAIT_PROFILES: list[PhilosophyTraits] = [
    PhilosophyTraits(Philosophy.EXPLICIT_GRAPH, 5, 5, 2, 5, 5, 3,
                       "Engineer", "Wrong edge logic in router function"),
    PhilosophyTraits(Philosophy.CONVERSATION, 2, 3, 5, 2, 2, 4,
                       "Manager / selector", "Chat runs too long; speaker loops"),
    PhilosophyTraits(Philosophy.TASK_DECOMPOSITION, 4, 4, 3, 4, 4, 5,
                       "Task author", "Task boundaries mis-specified"),
    PhilosophyTraits(Philosophy.SUPERVISOR, 3, 4, 4, 3, 3, 4,
                       "Supervisor agent", "Supervisor picks wrong worker"),
    PhilosophyTraits(Philosophy.PLAN_EXECUTE, 4, 4, 3, 4, 4, 3,
                       "Planner then executor", "Plan omits a required step"),
]

print(f"{'Philosophy':<22} {'Det':<4} {'Aud':<4} {'Flex':<4} {'Cost':<4} {'Debug':<4} Control")
print("-" * 72)
for t in TRAIT_PROFILES:
    print(f"{t.name.value:<22} {t.determinism:<4} {t.auditability:<4} {t.flexibility:<4} {t.predictable_cost:<4} {t.debuggability:<4} {t.control_locus}")

---

## 4. Shared Agents and Result Type

All philosophies invoke the same three specialists so comparisons are fair.

In [ ]:
@dataclass
class ReleaseState:
    goal: str
    artifacts: dict[str, Any] = field(default_factory=dict)
    messages: list[dict[str, str]] = field(default_factory=list)
    routing_log: list[str] = field(default_factory=list)

    def log(self, speaker: str, content: str) -> None:
        self.messages.append({"speaker": speaker, "content": content})

    def route(self, decision: str) -> None:
        self.routing_log.append(decision)


def researcher(state: ReleaseState) -> ReleaseState:
    facts = {"version": "2.4.0", "changes": [c["item"] for c in CHANGELOG], "standards": RUNBOOKS[0]["text"]}
    state.artifacts["research"] = facts
    state.log("researcher", f"Collected {len(facts['changes'])} changes")
    return state


def writer(state: ReleaseState) -> ReleaseState:
    facts = state.artifacts.get("research", {})
    draft = (
        f"Release v{facts.get('version', '?')} is now available.\n\n"
        f"What's new: " + "; ".join(facts.get("changes", [])) + ".\n\n"
        f"Breaking changes: Legacy v1 webhook format is deprecated.\n\n"
        f"Questions? Contact #releases."
    )
    state.artifacts["draft"] = draft
    state.log("writer", f"Draft ({len(draft.split())} words)")
    return state


def critic(state: ReleaseState) -> ReleaseState:
    draft = state.artifacts.get("draft", "")
    issues = []
    if "v2.4" not in draft:
        issues.append("missing version")
    if "Breaking changes" not in draft:
        issues.append("missing breaking section")
    if "#releases" not in draft:
        issues.append("missing support channel")
    approved = not issues
    state.artifacts["approved"] = approved
    state.log("critic", "APPROVED" if approved else f"REJECTED: {', '.join(issues)}")
    return state


AGENTS: dict[str, Callable[[ReleaseState], ReleaseState]] = {
    "researcher": researcher, "writer": writer, "critic": critic,
}


@dataclass
class OrchestrationResult:
    philosophy: Philosophy
    approved: bool
    agent_invocations: int
    routing_decisions: int
    routing_log: list[str]
    simulated_llm_calls: int
    draft_preview: str = ""


print("Agents ready:", list(AGENTS.keys()))

---

## 5. Philosophy 1 — Explicit Graph

**Belief:** Control flow is **infrastructure**. The engineer draws the graph; agents are nodes. Routing is a **pure function** of shared state.

| Strength | Weakness |
|----------|----------|
| Highest auditability | Rigid — new paths need code changes |
| Deterministic replay | Poor fit for open-ended exploration |
| Easy compliance story | Graph complexity grows with branches |

**Maps to:** LangGraph `StateGraph`, Step Functions, plain state machines.

In [ ]:
class ExplicitGraphOrchestrator:
    philosophy = Philosophy.EXPLICIT_GRAPH

    def route_after_review(self, state: ReleaseState) -> str:
        state.route("graph_edge: review → " + ("END" if state.artifacts.get("approved") else "write"))
        return "END" if state.artifacts.get("approved") else "write"

    def run(self, goal: str) -> OrchestrationResult:
        state = ReleaseState(goal=goal)
        invocations = 0
        llm_calls = 0  # graph itself uses 0 orchestration LLM calls

        for node in ("researcher", "writer", "critic"):
            state.route(f"graph_edge: START → {node}" if invocations == 0 else f"graph_edge: → {node}")
            state = AGENTS[node](state)
            invocations += 1
            llm_calls += 1  # each agent node = 1 LLM call in production

        nxt = self.route_after_review(state)
        if nxt != "END":
            state = AGENTS["writer"](state)
            invocations += 1
            llm_calls += 1
            state = AGENTS["critic"](state)
            invocations += 1
            llm_calls += 1

        return OrchestrationResult(
            philosophy=self.philosophy,
            approved=bool(state.artifacts.get("approved")),
            agent_invocations=invocations,
            routing_decisions=len(state.routing_log),
            routing_log=state.routing_log,
            simulated_llm_calls=llm_calls,
            draft_preview=state.artifacts.get("draft", "")[:120],
        )


g = ExplicitGraphOrchestrator().run("v2.4.0 announcement")
print(f"Graph: approved={g.approved} invocations={g.agent_invocations} routing={g.routing_decisions}")
print("Edges:", g.routing_log[:4])

---

## 6. Philosophy 2 — Emergent Conversation

**Belief:** Agents **talk**; coordination emerges from dialogue. A **manager** (or round-robin rule) picks the next speaker based on chat history.

| Strength | Weakness |
|----------|----------|
| Natural for brainstorming | Transcript grows quickly |
| Flexible turn order | Hard to bound cost |
| Good human-in-the-loop UX | Routing can be opaque |

**Maps to:** AutoGen `GroupChat`, multi-agent chat UIs.

In [ ]:
class ConversationOrchestrator:
    philosophy = Philosophy.CONVERSATION
    roster = ["researcher", "writer", "critic"]

    def select_speaker(self, turn: int, state: ReleaseState) -> str | None:
        if turn < len(self.roster):
            speaker = self.roster[turn]
            state.route(f"manager: select {speaker} (round {turn})")
            return speaker
        if not state.artifacts.get("approved") and state.artifacts.get("revisions", 0) < 1:
            state.artifacts["revisions"] = 1
            state.route("manager: critic rejected → select writer for revision")
            return "writer"
        state.route("manager: TERMINATE")
        return None

    def run(self, goal: str) -> OrchestrationResult:
        state = ReleaseState(goal=goal)
        state.log("user_proxy", goal)
        invocations = 0
        llm_calls = 1  # manager LLM per turn in full AutoGen
        turn = 0

        while True:
            speaker = self.select_speaker(turn, state)
            if speaker is None:
                break
            state = AGENTS[speaker](state)
            invocations += 1
            llm_calls += 1
            turn += 1
            if speaker == "writer" and turn > len(self.roster):
                state.route("manager: post-revision → select critic")
                state = AGENTS["critic"](state)
                invocations += 1
                llm_calls += 1
                turn += 1

        return OrchestrationResult(
            philosophy=self.philosophy,
            approved=bool(state.artifacts.get("approved")),
            agent_invocations=invocations,
            routing_decisions=len(state.routing_log),
            routing_log=state.routing_log,
            simulated_llm_calls=llm_calls,
            draft_preview=state.artifacts.get("draft", "")[:120],
        )


c = ConversationOrchestrator().run("v2.4.0 announcement")
print(f"Conversation: approved={c.approved} invocations={c.agent_invocations} llm_calls≈{c.simulated_llm_calls}")
for r in c.routing_log:
    print(f"  {r}")

---

## 7. Philosophy 3 — Task Decomposition

**Belief:** Work is a **list of tasks** with owners. Context flows forward through task outputs. The process type (sequential, hierarchical) is declared upfront.

| Strength | Weakness |
|----------|----------|
| Readable for business users | Less dynamic rerouting |
| Clear deliverables per task | Task boundaries can be wrong |
| Fast to prototype crews | Cross-task debugging |

**Maps to:** CrewAI `Crew` + `Task` + `Process.sequential`.

In [ ]:
@dataclass
class WorkTask:
    id: str
    description: str
    agent: str
    depends_on: list[str] = field(default_factory=list)


class TaskDecompositionOrchestrator:
    philosophy = Philosophy.TASK_DECOMPOSITION

    def __init__(self) -> None:
        self.task_queue = [
            WorkTask("t1", "Research v2.4.0 changelog and comm standards", "researcher"),
            WorkTask("t2", "Draft customer announcement from research", "writer", ["t1"]),
            WorkTask("t3", "Review draft against style guide", "critic", ["t2"]),
        ]

    def run(self, goal: str) -> OrchestrationResult:
        state = ReleaseState(goal=goal)
        completed: set[str] = set()
        invocations = 0
        llm_calls = 0

        for task in self.task_queue:
            if not all(dep in completed for dep in task.depends_on):
                state.route(f"task_queue: BLOCKED {task.id} — unmet deps {task.depends_on}")
                continue
            state.route(f"task_queue: execute {task.id} → {task.agent}")
            state = AGENTS[task.agent](state)
            completed.add(task.id)
            invocations += 1
            llm_calls += 1

        return OrchestrationResult(
            philosophy=self.philosophy,
            approved=bool(state.artifacts.get("approved")),
            agent_invocations=invocations,
            routing_decisions=len(state.routing_log),
            routing_log=state.routing_log,
            simulated_llm_calls=llm_calls,
            draft_preview=state.artifacts.get("draft", "")[:120],
        )


t = TaskDecompositionOrchestrator().run("v2.4.0 announcement")
print(f"Task queue: approved={t.approved} tasks_run={t.agent_invocations}")
for r in t.routing_log:
    print(f"  {r}")

---

## 8. Philosophy 4 — Supervisor Routing

**Belief:** A **supervisor** meta-agent examines state and **delegates** to the right worker each turn. Unlike a fixed graph, the supervisor can choose differently per run — but within a worker allow-list.

| Strength | Weakness |
|----------|----------|
| Adaptive delegation | Extra LLM call per turn |
| Clear worker specialization | Supervisor errors cascade |
| Good for dynamic task intake | Harder to prove exact path |

**Maps to:** LangGraph supervisor pattern, hierarchical AutoGen, "router" nodes.

In [ ]:
class SupervisorOrchestrator:
    philosophy = Philosophy.SUPERVISOR
    workers = {"researcher", "writer", "critic"}

    def supervisor_decide(self, state: ReleaseState, turn: int) -> str | None:
        """Simulated supervisor LLM — inspects artifacts, picks worker."""
        if "research" not in state.artifacts:
            choice = "researcher"
        elif "draft" not in state.artifacts:
            choice = "writer"
        elif "approved" not in state.artifacts:
            choice = "critic"
        elif not state.artifacts.get("approved") and state.artifacts.get("retries", 0) < 1:
            state.artifacts["retries"] = 1
            choice = "writer"
        else:
            state.route("supervisor: FINISH — goal met or max retries")
            return None
        state.route(f"supervisor: delegate → {choice} (turn {turn})")
        return choice

    def run(self, goal: str) -> OrchestrationResult:
        state = ReleaseState(goal=goal)
        invocations = 0
        llm_calls = 0
        turn = 0
        max_turns = 8

        while turn < max_turns:
            llm_calls += 1  # supervisor reasoning call
            worker = self.supervisor_decide(state, turn)
            if worker is None:
                break
            state = AGENTS[worker](state)
            invocations += 1
            llm_calls += 1  # worker LLM call
            turn += 1

        return OrchestrationResult(
            philosophy=self.philosophy,
            approved=bool(state.artifacts.get("approved")),
            agent_invocations=invocations,
            routing_decisions=len(state.routing_log),
            routing_log=state.routing_log,
            simulated_llm_calls=llm_calls,
            draft_preview=state.artifacts.get("draft", "")[:120],
        )


s = SupervisorOrchestrator().run("v2.4.0 announcement")
print(f"Supervisor: approved={s.approved} invocations={s.agent_invocations} llm≈{s.simulated_llm_calls}")
for r in s.routing_log:
    print(f"  {r}")

---

## 9. Philosophy 5 — Plan-and-Execute

**Belief:** Separate **planning** from **execution**. A planner LLM emits an ordered step list; an executor runs it with minimal further reasoning. Re-plan only on failure.

| Strength | Weakness |
|----------|----------|
| Cheaper execution phase | Plan may be wrong or incomplete |
| Clear upfront intent | Re-planning adds latency |
| Good for known task templates | Poor for highly dynamic environments |

**Maps to:** LangChain plan-and-execute agents, baby-AGI style loops, planner nodes in graphs.

In [ ]:
class PlanAndExecuteOrchestrator:
    philosophy = Philosophy.PLAN_EXECUTE

    def plan(self, goal: str) -> list[str]:
        """Simulated planner LLM — returns agent sequence for goal."""
        return ["researcher", "writer", "critic"]

    def run(self, goal: str) -> OrchestrationResult:
        state = ReleaseState(goal=goal)
        state.route("planner: generate step list (1 LLM call)")
        plan = self.plan(goal)
        state.route(f"planner: plan = {' → '.join(plan)}")
        llm_calls = 1  # planning phase
        invocations = 0

        for i, step in enumerate(plan):
            state.route(f"executor: step {i+1}/{len(plan)} → {step}")
            state = AGENTS[step](state)
            invocations += 1
            llm_calls += 1

        if not state.artifacts.get("approved"):
            state.route("executor: plan failed review → REPLAN")
            replan = ["writer", "critic"]
            llm_calls += 1  # replan call
            for step in replan:
                state.route(f"executor: replan → {step}")
                state = AGENTS[step](state)
                invocations += 1
                llm_calls += 1

        return OrchestrationResult(
            philosophy=self.philosophy,
            approved=bool(state.artifacts.get("approved")),
            agent_invocations=invocations,
            routing_decisions=len(state.routing_log),
            routing_log=state.routing_log,
            simulated_llm_calls=llm_calls,
            draft_preview=state.artifacts.get("draft", "")[:120],
        )


p = PlanAndExecuteOrchestrator().run("v2.4.0 announcement")
print(f"Plan-execute: approved={p.approved} steps={p.agent_invocations} llm≈{p.simulated_llm_calls}")
for r in p.routing_log:
    print(f"  {r}")

---

## 10. Head-to-Head Results

Same goal, five philosophies:

In [ ]:
ORCHESTRATORS = [
    ExplicitGraphOrchestrator(),
    ConversationOrchestrator(),
    TaskDecompositionOrchestrator(),
    SupervisorOrchestrator(),
    PlanAndExecuteOrchestrator(),
]

GOAL = "Produce v2.4.0 customer release announcement"
results = [o.run(GOAL) for o in ORCHESTRATORS]

print(f"{'Philosophy':<22} {'OK':<6} {'Agents':<8} {'Routes':<8} {'LLM≈':<6}")
print("-" * 55)
for r in results:
    print(f"{r.philosophy.value:<22} {str(r.approved):<6} {r.agent_invocations:<8} {r.routing_decisions:<8} {r.simulated_llm_calls:<6}")

---

## 11. Centralized vs Decentralized Control

| Style | Philosophies | Who routes? |
|-------|--------------|-------------|
| **Centralized (engineer)** | Explicit graph, Task decomposition | Code / task list at build time |
| **Centralized (orchestrator LLM)** | Supervisor, Plan-and-execute | Supervisor or planner model |
| **Decentralized (emergent)** | Conversation | Manager reacts to chat; agents can pitch next steps |

**Production bias:** Start centralized (engineer-owned) for compliance; add orchestrator LLM routing only where inputs are too variable for static edges.

---

## 12. State-Centric vs Message-Centric

| Model | Primary memory | Philosophies |
|-------|----------------|--------------|
| **State-centric** | Typed shared object (`artifacts`, flags) | Graph, Supervisor, Plan-execute |
| **Message-centric** | Chat transcript | Conversation |
| **Artifact-centric** | Task outputs chained as context | Task decomposition |

Hybrid systems often **project** messages into state (or vice versa) at step boundaries — the philosophy tells you which is source of truth.

In [ ]:
STATE_PHILOSOPHIES = [Philosophy.EXPLICIT_GRAPH, Philosophy.SUPERVISOR, Philosophy.PLAN_EXECUTE]
MESSAGE_PHILOSOPHIES = [Philosophy.CONVERSATION]
ARTIFACT_PHILOSOPHIES = [Philosophy.TASK_DECOMPOSITION]

def primary_memory(p: Philosophy) -> str:
    if p in STATE_PHILOSOPHIES:
        return "shared state / artifacts"
    if p in MESSAGE_PHILOSOPHIES:
        return "chat transcript"
    return "task output chain"


for p in Philosophy:
    print(f"{p.value:<22} memory: {primary_memory(p)}")

---

## 13. Cost and Latency Trade-offs

Simulated LLM call counts from section 10 illustrate a recurring pattern:

| Philosophy | Orchestration overhead | Agent calls | Total ≈ |
|------------|------------------------|-------------|--------|
| Explicit graph | 0 supervisor calls | 3–5 | Lowest overhead |
| Task decomposition | 0 | 3 | Low |
| Plan-and-execute | 1–2 plan calls | 3–5 | Medium |
| Supervisor | 1 per turn | 3–5 | Higher |
| Conversation | 1 manager per turn | 3–5 | Higher |

**Rule:** Every LLM-based routing decision adds **latency + cost + failure mode**. Pay for it only when static routing cannot work.

---

## 14. Hybrid Philosophies in Production

Real systems **compose** philosophies:

```
Explicit graph shell
  ├── classify (deterministic)
  ├── plan-and-execute (planner → executor loop)
  ├── supervisor (dynamic delegation inside one node)
  └── human_gate (deterministic)
```

| Hybrid | Why combine |
|--------|-------------|
| Graph + Supervisor | Fixed compliance path; adaptive worker choice inside |
| Graph + Plan-execute | Planner only in "research" node |
| Task crew + Graph gate | Crew drafts; graph handles publish approval |
| Conversation + Graph cap | Chat UX with `max_rounds` enforced by outer graph |

In [ ]:
class HybridOrchestrator:
    """Graph shell → plan-execute core → graph publish gate."""

    def run(self, goal: str) -> OrchestrationResult:
        state = ReleaseState(goal=goal)
        routing: list[str] = []

        # Graph: fixed classify
        routing.append("graph: classify → release_announcement")
        state.artifacts["category"] = "release_announcement"

        # Plan-execute middle
        pe = PlanAndExecuteOrchestrator()
        pe_result = pe.run(goal)
        state.artifacts.update({"draft": pe_result.draft_preview, "approved": pe_result.approved})
        routing.append("hybrid: plan-execute phase complete")
        routing.extend(pe_result.routing_log[:3])

        # Graph: publish gate
        if state.artifacts.get("approved"):
            routing.append("graph: publish_gate → PUBLISH")
        else:
            routing.append("graph: publish_gate → BLOCKED")

        return OrchestrationResult(
            philosophy=Philosophy.EXPLICIT_GRAPH,
            approved=pe_result.approved,
            agent_invocations=pe_result.agent_invocations,
            routing_decisions=len(routing),
            routing_log=routing,
            simulated_llm_calls=pe_result.simulated_llm_calls,
            draft_preview=pe_result.draft_preview,
        )


hybrid = HybridOrchestrator().run(GOAL)
print(f"Hybrid: approved={hybrid.approved} routes={hybrid.routing_decisions}")
for r in hybrid.routing_log:
    print(f"  {r}")

---

## 15. Decision Worksheet

Answer these before picking a philosophy:

In [ ]:
DECISION_QUESTIONS = [
    ("Must regulators audit exact step order?", Philosophy.EXPLICIT_GRAPH),
    ("Is the product UX a multi-agent chat?", Philosophy.CONVERSATION),
    ("Is work already defined as role-owned deliverables?", Philosophy.TASK_DECOMPOSITION),
    ("Does intake vary but workers are stable specialists?", Philosophy.SUPERVISOR),
    ("Is there a known template with occasional replanning?", Philosophy.PLAN_EXECUTE),
]


def score_philosophies(answers: dict[str, bool]) -> dict[Philosophy, int]:
    scores: dict[Philosophy, int] = {p: 0 for p in Philosophy}
    for question, philosophy in DECISION_QUESTIONS:
        if answers.get(question, False):
            scores[philosophy] += 1
    return scores


sample_answers = {
    "Must regulators audit exact step order?": True,
    "Is the product UX a multi-agent chat?": False,
    "Is work already defined as role-owned deliverables?": True,
    "Does intake vary but workers are stable specialists?": False,
    "Is there a known template with occasional replanning?": True,
}

scores = score_philosophies(sample_answers)
winner = max(scores, key=scores.get)
print("Scores for sample release-announcement product:")
for p, s in sorted(scores.items(), key=lambda x: -x[1]):
    mark = " ← top" if p == winner else ""
    print(f"  {p.value:<22} {s}{mark}")

---

## 16. Failure Modes by Philosophy

| Philosophy | Typical failure | Detection |
|------------|-----------------|-----------|
| Explicit graph | Wrong conditional edge | Trace shows unexpected node |
| Conversation | Speaker loop / max rounds | Growing transcript, no termination |
| Task decomposition | Task missing dependency | Empty context in downstream task |
| Supervisor | Wrong worker selected | Artifact never populated |
| Plan-and-execute | Plan skips required step | Executor completes but goal unmet |

---

## 17. Quiz

1. What is the difference between a philosophy and a framework?
2. Which philosophy has the engineer own routing at deploy time?
3. Why does supervisor routing cost more than an explicit graph?
4. When is plan-and-execute a better fit than emergent conversation?
5. Name one productive hybrid composition.

---

## 18. Summary

| Philosophy | Control locus | Best when |
|------------|---------------|-----------|
| **Explicit graph** | Engineer | Auditability, compliance, fixed pipelines |
| **Conversation** | Manager / chat | Brainstorming, HITL chat UX |
| **Task decomposition** | Task author | Role-owned deliverables, crews |
| **Supervisor** | Supervisor LLM | Variable intake, stable specialists |
| **Plan-and-execute** | Planner then executor | Templated work with occasional replan |

| Takeaway | Detail |
|----------|--------|
| Philosophy before vendor | Same idea appears in multiple frameworks |
| Pay for LLM routing sparingly | Supervisor + conversation add calls per turn |
| Same agents, different routers | Specialists are stable; orchestration is the variable |
| Hybrids are normal | Graph shell + adaptive inner node is production-default |

Choose the philosophy that matches **who must own the next step** in your product — then pick the framework that expresses it cleanly.